In [0]:
from pyspark.sql import functions as F

In [0]:
violations = spark.table(
    "platform_monitoring.governance_violations"
)

In [0]:
alerts = (
    violations
    .withColumn(
        "alert_id",
        F.expr("uuid()")
    )
    .withColumn(
        "alert_time",
        F.current_timestamp()
    )
    .withColumn(
        "alert_status",
        F.lit("ACTIVE")
    )
    .withColumn(
        "alert_message",
        F.concat(
            F.lit("Governance Alert: "),
            F.col("object_name"),
            F.lit(" violated rule "),
            F.col("violation_type")
        )
    )
    .select(
        "alert_id",
        "object_name",
        "object_type",
        "severity",
        "alert_message",
        "alert_time",
        "alert_status"
    )
)

In [0]:
alerts.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(
    "platform_monitoring.alerts"
)

In [0]:
spark.table(
    "platform_monitoring.alerts"
).show(50, False)

+------------------------------------+------------------+-----------+--------+-------------------------------------------------------------------+--------------------------+------------+
|alert_id                            |object_name       |object_type|severity|alert_message                                                      |alert_time                |alert_status|
+------------------------------------+------------------+-----------+--------+-------------------------------------------------------------------+--------------------------+------------+
|416ef2a8-a8c5-4ffd-80ad-a597f70249f4|audit_logs        |TABLE      |HIGH    |Governance Alert: audit_logs violated rule STALE_TABLE             |2026-06-30 17:24:36.252714|ACTIVE      |
|85b28c5a-2728-467e-9950-80fbfdf2001b|shipments         |TABLE      |MEDIUM  |Governance Alert: shipments violated rule LOW_USAGE                |2026-06-30 17:24:36.252714|ACTIVE      |
|d30bd8a6-8ae8-4c93-b94a-344cd29ae25e|Inventory_Sync    |JOB     